# **NYC Taxi project**

## Setting some environment configs

In [1]:
import os
import sys

# Java
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# Force Spark to use same Python as Jupyter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


## Creating a spark session

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("NYCTaxi") \
    .getOrCreate()

sc = spark.sparkContext

## Optimiztion configs

In [3]:
sc.defaultParallelism


12

In [4]:

spark.conf.get( "spark.sql.shuffle.partitions" )

'200'

# **Analyzing New York City Taxi** 


## Reading Data with predefined schema

### df reapartitioned to 12 as the number of cores to maximize parallesim

In [5]:
# Reading Taxi data
file_path ="D:\Spark\proj\data\sample.csv"
schema = """
medallion STRING,
hack_license STRING,
vendor_id STRING,
rate_code INT,
store_and_fwd_flag STRING,
pickup_datetime TIMESTAMP,
dropoff_datetime TIMESTAMP,
passenger_count INT,
trip_time_in_secs INT,
trip_distance DOUBLE,
pickup_longitude DOUBLE,
pickup_latitude DOUBLE,
dropoff_longitude DOUBLE,
dropoff_latitude DOUBLE
"""

df = spark.read.csv(file_path, header=True, schema=schema).repartition(12)


## Selecting columns of interset for taxi dataset

In [6]:
taxi_df = df.select("medallion","pickup_datetime","dropoff_datetime","pickup_longitude","pickup_latitude","dropoff_longitude","dropoff_latitude")
taxi_df.show(10)

+--------------------+-------------------+-------------------+----------------+---------------+-----------------+----------------+
|           medallion|    pickup_datetime|   dropoff_datetime|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|
+--------------------+-------------------+-------------------+----------------+---------------+-----------------+----------------+
|4029B98D6CCE7239A...|2013-01-13 12:08:00|2013-01-13 12:16:00|      -73.991119|      40.750191|       -73.973724|       40.764168|
|0DF80FDB0E48E1A25...|2013-01-13 10:09:00|2013-01-13 10:14:00|      -73.990746|       40.74865|       -73.988487|       40.745537|
|DCE5517544F627101...|2013-01-13 02:07:00|2013-01-13 02:22:00|      -73.990211|      40.737114|       -73.953018|       40.772129|
|F1D1F07C119A09262...|2013-01-13 11:01:00|2013-01-13 11:05:00|      -73.969849|      40.784672|       -73.957458|       40.780312|
|0EC22AAF491A8BD91...|2013-01-13 10:52:00|2013-01-13 11:10:00|      -73.953217|    

In [7]:
taxi_df.rdd.getNumPartitions()

12

## Reading nyc-boroughs.geojson

In [8]:
import json
from shapely.geometry import shape, Point
with open("D:\\Spark\\proj\\data\\nyc-boroughs.geojson") as f:
    data = json.load(f)
features = []
for feat in data["features"]:
    polygon = shape(feat["geometry"])
    borough = feat["properties"]["borough"]
    code    = feat["properties"]["boroughCode"]
    features.append((code, polygon.area, borough, polygon))

## Sort: ascending boroughCode, descending area within each borough

In [9]:
features.sort(key=lambda x: (x[0], -x[1]))

##  Keep only what UDF actually needs: (borough_name, polygon)

In [10]:
clean_features = [(borough, polygon) for _, _, borough, polygon in features]

### Broadcast the boroughs

In [11]:
broadcast_boroughs = spark.sparkContext.broadcast(clean_features)

## Creating get_borough() UDF 

In [12]:
def get_borough(lon, lat):
    if lon is None or lat is None:
        return "Unknown"
    pt = Point(lon, lat)                       
    for borough_name, polygon in broadcast_boroughs.value:
        if polygon.contains(pt):
            return borough_name
    return "Unknown" 

## Creating get_borough() UDF in spark

In [13]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql import functions as F

get_borough_udf = F.udf(get_borough, StringType())

## Apply get_borough() UDF to the taxi DataFrame

In [14]:
mapped_taxi_df=( taxi_df.withColumn(
        "pickup_borough",
        get_borough_udf(F.col("pickup_longitude"), F.col("pickup_latitude"))
    ).withColumn(
        "dropoff_borough",
        get_borough_udf(F.col("dropoff_longitude"), F.col("dropoff_latitude"))
    )
     .drop("pickup_longitude","pickup_latitude","dropoff_longitude","dropoff_latitude")          
               )
mapped_taxi_df.show(10)

+--------------------+-------------------+-------------------+--------------+---------------+
|           medallion|    pickup_datetime|   dropoff_datetime|pickup_borough|dropoff_borough|
+--------------------+-------------------+-------------------+--------------+---------------+
|4029B98D6CCE7239A...|2013-01-13 12:08:00|2013-01-13 12:16:00|     Manhattan|      Manhattan|
|0DF80FDB0E48E1A25...|2013-01-13 10:09:00|2013-01-13 10:14:00|     Manhattan|      Manhattan|
|DCE5517544F627101...|2013-01-13 02:07:00|2013-01-13 02:22:00|     Manhattan|      Manhattan|
|F1D1F07C119A09262...|2013-01-13 11:01:00|2013-01-13 11:05:00|     Manhattan|      Manhattan|
|0EC22AAF491A8BD91...|2013-01-13 10:52:00|2013-01-13 11:10:00|     Manhattan|      Manhattan|
|1102DFAC80502A90E...|2013-01-13 10:08:00|2013-01-13 10:14:00|     Manhattan|      Manhattan|
|840CE7F507E6300CA...|2013-01-13 11:51:00|2013-01-13 12:14:00|     Manhattan|      Manhattan|
|D409FE36415B23FDA...|2013-01-13 09:46:00|2013-01-13 10:00:0

## ** **Data Cleansing** **

### First we drop the "unknown" pickup_borough and dropoff_borough resulting from get_borough UDF

In [15]:
c_mapped_taxi_df = ( mapped_taxi_df
                    .filter( (F.col("pickup_borough")  != "Unknown") &  (F.col("dropoff_borough") != "Unknown") )
                   )

### We need to get red of dirty data where dropoff_datetime is less than pickup_datetime and also exclude outliers
### We will compute a Duration column which = dropoff_datetime - pickup_datetime


In [16]:
# unix_timestamp generates output in seconds 
# to get duration in hours output divided by 3600
final_taxi_df = ( c_mapped_taxi_df
                 .withColumn("duration",(F.unix_timestamp("dropoff_datetime")-F.unix_timestamp("pickup_datetime") )/3600)
                )    

### Exploring duration data

In [17]:
final_taxi_df.select(F.max("duration"),F.min("duration")).show()

+-------------+-------------+
|max(duration)|min(duration)|
+-------------+-------------+
|         2.55|          0.0|
+-------------+-------------+



### After computing Duration column, we can drop records where duration is negative or greater than a threshold like 4 hours  


In [18]:
threshold = 4 
cleaned_taxi_df = final_taxi_df.where( (F.col("duration") < threshold) & (F.col("duration") > 0) )

In [19]:
cleaned_taxi_df.where( (F.col("duration") > threshold) | (F.col("duration") <= 0) ).show()

+---------+---------------+----------------+--------------+---------------+--------+
|medallion|pickup_datetime|dropoff_datetime|pickup_borough|dropoff_borough|duration|
+---------+---------------+----------------+--------------+---------------+--------+
+---------+---------------+----------------+--------------+---------------+--------+



##






##



## ** **Data Aggregation and Bussiness insights** **


### 1- The avgerage time per taxi to find its next trip per destination brought 

### Defining window

In [20]:
from pyspark.sql import Window
w = Window.partitionBy("medallion").orderBy("pickup_datetime")
w_taxi_df = ( cleaned_taxi_df
             .withColumn("prev_pickup",  F.lag("pickup_datetime").over(w))
             .withColumn("idle_time_mins",  (F.unix_timestamp("pickup_datetime")-F.unix_timestamp("prev_pickup"))/60)
             .repartition(12) )

In [21]:
( w_taxi_df
.select("medallion","pickup_datetime","dropoff_borough","prev_pickup","idle_time_mins")
.orderBy("medallion","pickup_datetime")
.show()
)

+--------------------+-------------------+---------------+-------------------+--------------+
|           medallion|    pickup_datetime|dropoff_borough|        prev_pickup|idle_time_mins|
+--------------------+-------------------+---------------+-------------------+--------------+
|000318C2E3E638158...|2013-01-13 04:41:00|      Manhattan|               null|          null|
|000318C2E3E638158...|2013-01-13 12:24:00|      Manhattan|2013-01-13 04:41:00|         463.0|
|000318C2E3E638158...|2013-01-13 12:39:00|      Manhattan|2013-01-13 12:24:00|          15.0|
|000318C2E3E638158...|2013-01-13 12:51:00|      Manhattan|2013-01-13 12:39:00|          12.0|
|000318C2E3E638158...|2013-01-13 13:08:00|      Manhattan|2013-01-13 12:51:00|          17.0|
|000318C2E3E638158...|2013-01-13 13:24:00|      Manhattan|2013-01-13 13:08:00|          16.0|
|000318C2E3E638158...|2013-01-13 13:38:00|      Manhattan|2013-01-13 13:24:00|          14.0|
|000318C2E3E638158...|2013-01-13 14:14:00|      Manhattan|20

### Cleaning window dataframe

In [22]:
w_taxi_df.rdd.getNumPartitions()

12

In [23]:
four_hours = 4
c_w_taxi_df = (w_taxi_df.filter(
    F.col("idle_time_mins").isNotNull() &
    (F.col("idle_time_mins") > 0) &
    (F.col("idle_time_mins") <= four_hours  * 60 ))
              )

### **Final Results**

In [24]:
query1 = ( c_w_taxi_df.groupBy("medallion","dropoff_borough") 
                .agg(
                    F.avg("idle_time_mins").alias("avg_idle_mins"),
                ) 
                .orderBy("avg_idle_mins")
         )  
query1.orderBy("medallion").show()

+--------------------+---------------+------------------+
|           medallion|dropoff_borough|     avg_idle_mins|
+--------------------+---------------+------------------+
|000318C2E3E638158...|      Manhattan|20.636363636363637|
|002B4CFC5B8920A87...|      Manhattan| 34.46153846153846|
|002B4CFC5B8920A87...|         Queens|              10.0|
|002E3B405B6ABEA23...|      Manhattan|            42.375|
|002E3B405B6ABEA23...|         Queens|              91.0|
|0030AD2648D81EE87...|       Brooklyn|              21.0|
|0035520A854E4F276...|       Brooklyn|             100.0|
|0035520A854E4F276...|      Manhattan|25.363636363636363|
|0036961468659D0BF...|      Manhattan|29.764705882352942|
|003889E315BFDD985...|      Manhattan|             28.75|
|0038EF45118925A51...|      Manhattan|              22.0|
|003D87DB553C6F00F...|      Manhattan|16.541666666666668|
|003EEA559FA618008...|      Manhattan|             42.55|
|003EEA559FA618008...|         Queens|              14.0|
|0053334C798EC

## 2- The number of trips that started and ended within the same borough

### Total Count

In [25]:
( cleaned_taxi_df
 .where(F.col("pickup_borough")==F.col("dropoff_borough"))
 .count()
)

85945

### count per borough

In [26]:
( cleaned_taxi_df
 .where(F.col("pickup_borough")==F.col("dropoff_borough"))
 .groupBy("pickup_borough")
 .agg(F.count("*").alias("num_trips"))
 .orderBy("num_trips", ascending=False)
 .show()
)

+--------------+---------+
|pickup_borough|num_trips|
+--------------+---------+
|     Manhattan|    83464|
|        Queens|     1369|
|      Brooklyn|     1062|
|         Bronx|       49|
| Staten Island|        1|
+--------------+---------+



## 3- The number of trips that start in one borough and ended in another borough

### Total conut

In [27]:
( cleaned_taxi_df
 .where(F.col("pickup_borough")!=F.col("dropoff_borough"))
 .count() 
)

11431

### Count ber borough

In [28]:
( cleaned_taxi_df
 .where(F.col("pickup_borough")!=F.col("dropoff_borough"))
 .groupBy("pickup_borough")
 .agg(F.count("*").alias("num_trips"))
 .orderBy("num_trips", ascending=False)
 .show()
)

+--------------+---------+
|pickup_borough|num_trips|
+--------------+---------+
|     Manhattan|     6119|
|        Queens|     4396|
|      Brooklyn|      888|
|         Bronx|       27|
| Staten Island|        1|
+--------------+---------+

